#### Homework

In [ ]:
!pip install dlt[duckdb]

In [2]:
!dlt --version

dlt 1.6.1


In [19]:
import dlt
from dlt.sources.helpers.rest_client import RESTClient
from dlt.sources.helpers.rest_client.paginators import PageNumberPaginator
from typing import Dict, Generator
from dlt.sources.helpers import requests

In [ ]:
@dlt.resource(
    write_disposition="replace",
    primary_key="rides_id"
)
def ny_taxi(client: RESTClient) -> Generator[Dict, None, None]:
    """
    Extract NY Taxi data from the API using pagination
    """
    paginator = PageNumberPaginator(
        page_size=10,  # Number of items per page
        start=1,  # Starting page number
        page_param="page",  # Parameter name for page number in the URL
        size_param="page_size",  # Parameter name for page size in the URL
        items_path="result"  # JSON path to the array of items in the response
    )

    # Get paginated data
    for page in client.get_pages(
        url="",  # Base URL is already set in the client
        paginator=paginator
    ):
        yield from page["result"]

def init_pipeline():
    """Initialize the pipeline with the API client"""
    # Create REST client with base URL
    client = RESTClient(
        base_url="https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api"
    )

    # Create pipeline
    pipeline = dlt.pipeline(
        pipeline_name="ny_taxi_pipeline",
        destination="duckdb",
        dataset_name="ny_taxi_data"
    )

    # Run pipeline with the ny_taxi resource
    load_info = pipeline.run(ny_taxi(client=client))
    print(load_info)

    return pipeline

def query_data(pipeline):
    """Query the loaded data using DuckDB"""
    # Connect to DuckDB
    conn = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")
    
    # Set search path to the dataset
    conn.sql(f"SET search_path = '{pipeline.dataset_name}'")
    
    # Describe the dataset
    print("Dataset Schema:")
    print(conn.sql("DESCRIBE").df())
    
    # Show sample data
    print("\nSample Data:")
    print(conn.sql("SELECT * FROM ny_taxi LIMIT 5").df())
    
    return conn

In [ ]:
if __name__ == "__main__":
    # Run the pipeline
    pipeline = init_pipeline()
    
    # Query the data
    conn = query_data(pipeline)

In [23]:
def ny_taxi() -> Generator[Dict, None, None]:
    """
    Extract NY Taxi data from the API using pagination.
    """
    base_url = "https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api"
    page = 1
    page_size = 10
    
    while True:
        # Make request with current page parameters
        response = requests.get(
            base_url,
            params={
                "page": page,
                "page_size": page_size
            }
        )
        
        # Check if request was successful
        response.raise_for_status()
        
        # Get the records from response
        records = response.json()
        
        # If no more records or empty list, break the loop
        if not records:
            break
            
        # Yield each record
        for record in records:
            yield record
            
        # Move to next page
        page += 1

# Create the pipeline
pipeline = dlt.pipeline(
    pipeline_name="ny_taxi_pipeline",
    destination="duckdb",
    dataset_name="ny_taxi_data"
)

# Run the pipeline
load_info = pipeline.run(ny_taxi())
print(load_info)

Pipeline ny_taxi_pipeline load step completed in 1.29 seconds
1 load package(s) were loaded to destination duckdb and into dataset ny_taxi_data
The duckdb destination used duckdb:////workspaces/de2025/Workshop-1/ny_taxi_pipeline.duckdb location to store data
Load package 1739924542.6833766 is LOADED and contains no failed jobs


In [ ]:
import duckdb

# Connect to the database
conn = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")

# Set search path to the dataset
conn.sql(f"SET search_path = '{pipeline.dataset_name}'")


# Show the data
print("\nData in DuckDB:")
print(conn.sql("SELECT COUNT(*) as total_records FROM ny_taxi").df())
print("\nSample of records:")
print(conn.sql("DESCRIBE").df())


Data in DuckDB:
   total_records
0          10000

Sample of records:
           database        schema                 name  \
0  ny_taxi_pipeline  ny_taxi_data           _dlt_loads   
1  ny_taxi_pipeline  ny_taxi_data  _dlt_pipeline_state   
2  ny_taxi_pipeline  ny_taxi_data         _dlt_version   
3  ny_taxi_pipeline  ny_taxi_data              ny_taxi   

                                        column_names  \
0  [load_id, schema_name, status, inserted_at, sc...   
1  [version, engine_version, pipeline_name, state...   
2  [version, engine_version, inserted_at, schema_...   
3  [end_lat, end_lon, fare_amt, passenger_count, ...   

                                        column_types  temporary  
0  [VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...      False  
1  [BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...      False  
2  [BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...      False  
3  [DOUBLE, DOUBLE, DOUBLE, BIGINT, VARCHAR, DOUB...      False  


In [25]:
print("\nTables in the database:")
tables = conn.sql("""
    SELECT table_schema, table_name, table_type 
    FROM information_schema.tables 
    WHERE table_schema = 'ny_taxi_data'
""").df()
print(tables)


Tables in the database:
   table_schema           table_name  table_type
0  ny_taxi_data              ny_taxi  BASE TABLE
1  ny_taxi_data           _dlt_loads  BASE TABLE
2  ny_taxi_data  _dlt_pipeline_state  BASE TABLE
3  ny_taxi_data         _dlt_version  BASE TABLE


### Question-2
Total tables created are four (4)

### Question-3
Total number of records are 10000

In [27]:
df = pipeline.dataset(dataset_type="default").ny_taxi.df()
df

,end_lat,end_lon,fare_amt,passenger_count,payment_type,start_lat,start_lon,tip_amt,tolls_amt,total_amt,trip_distance,trip_dropoff_date_time,trip_pickup_date_time,surcharge,vendor_name,_dlt_load_id,_dlt_id,store_and_forward
0,40.742963,-73.980072,45.0,1,Credit,40.641525,-73.787442,9.0,4.15,58.15,17.52,2009-06-14 23:48:00+00:00,2009-06-14 23:23:00+00:00,0.0,VTS,1739924542.6833766,7AqYYG8MhjJGog,NaN
1,40.740187,-74.005698,6.5,1,Credit,40.722065,-74.009767,1.0,0.00,8.50,1.56,2009-06-18 17:43:00+00:00,2009-06-18 17:35:00+00:00,1.0,VTS,1739924542.6833766,nNQR2z4V/29cuA,NaN
2,40.718043,-74.004745,12.5,5,Credit,40.761945,-73.983038,2.0,0.00,15.50,3.37,2009-06-10 18:27:00+00:00,2009-06-10 18:08:00+00:00,1.0,VTS,1739924542.6833766,cFvddOjFNrVvvA,NaN
3,40.739637,-73.985233,4.9,1,CASH,40.749802,-73.992247,0.0,0.00,5.40,1.11,2009-06-14 23:58:00+00:00,2009-06-14 23:54:00+00:00,0.5,VTS,1739924542.6833766,TNkOIVwqU7dCvg,NaN
4,40.730032,-73.852693,25.7,1,CASH,40.776825,-73.949233,0.0,4.15,29.85,11.09,2009-06-13 13:23:00+00:00,2009-06-13 13:01:00+00:00,0.0,VTS,1739924542.6833766,LXW2Ugaqz+rimg,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,40.783522,-73.970690,5.7,1,CASH,40.778560,-73.953660,0.0,0.00,5.70,1.16,2009-06-19 11:28:00+00:00,2009-06-19 11:22:00+00:00,0.0,VTS,1739924542.6833766,owIgIn4UH9tGPA,NaN
9996,40.777200,-73.964197,4.1,1,CASH,40.779800,-73.974297,0.0,0.00,4.10,0.89,2009-06-17 07:43:00+00:00,2009-06-17 07:41:00+00:00,0.0,VTS,1739924542.6833766,V1sLzLjSgNWPfg,NaN
9997,40.780172,-73.957617,6.1,1,CASH,40.788388,-73.976758,0.0,0.00,6.10,1.30,2009-06-19 11:46:00+00:00,2009-06-19 11:39:00+00:00,0.0,VTS,1739924542.6833766,Vb5x6RgjjAWzTg,NaN
9998,40.777342,-73.957242,5.7,1,CASH,40.773828,-73.956690,0.0,0.00,6.20,0.97,2009-06-17 04:19:00+00:00,2009-06-17 04:13:00+00:00,0.5,VTS,1739924542.6833766,Z5I19V2O5r0Ynw,NaN


### Question-4
the average trip duration is 12.3049

In [28]:
with pipeline.sql_client() as client:
    res = client.execute_sql(
            """
            SELECT
            AVG(date_diff('minute', trip_pickup_date_time, trip_dropoff_date_time))
            FROM ny_taxi;
            """
        )
    # Prints column values of the first row
    print(res)

[(12.3049,)]
